# 1. Dobór metody i dane graniczne
Notebook prowadzącego. Pokazuje, dlaczego najpierw diagnozujemy problem, a następnie budujemy minimalne pary bez leakage. Nie otwiera chronionych splitów.

In [ ]:
from pathlib import Path
import json

def project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Uruchom notebook wewnątrz repozytorium peft')

ROOT = project_root()
ROOT

## Drzewo decyzji
- zmienna wiedza i cytaty → RAG,
- stabilny format lub etykiety → prompt/few-shot, potem PEFT,
- jawna reguła → kod deterministyczny,
- szeroka luka językowa/domenowa → rozważ continued pretraining.

In [ ]:
scenarios = [
    ('Zmienna procedura z cytatem', 'RAG'),
    ('Stabilny kontrakt pięciu statusów', 'PEFT/SFT'),
    ('Próg: abs(actual-expected) > 5', 'kod deterministyczny'),
    ('Nowe słownictwo w dużym korpusie', 'continued pretraining + RAG/SFT'),
]
for problem, method in scenarios:
    print(f'{problem:42} → {method}')

## Minimalna para PASS → WARN
Poniższe rekordy mają wspólny `group_id`. Zmieniona zostaje jedna przesłanka: różnica 0,4 vs 0,8 mln PLN.

In [ ]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

boundary = read_jsonl(ROOT / 'data/splits/boundary_train.jsonl')
pair = [row for row in boundary if row['group_id'] == 'boundary-pass_warn-f000']
for row in pair:
    evidence = row['input']['sources'][1]['content']
    status = row['expected_output']['status']
    print(row['case_id'], status, '—', evidence)

In [ ]:
# Kontrola leakage: każda rodzina powinna należeć tylko do jednego splitu.
paths = list((ROOT / 'data/splits').glob('boundary_*.jsonl'))
owners = {}
for path in paths:
    for row in read_jsonl(path):
        owners.setdefault(row['group_id'], set()).add(path.stem)
leaks = {group: splits for group, splits in owners.items() if len(splits) > 1}
print('Rodziny:', len(owners), '| leakage:', len(leaks))
assert not leaks

## Pytanie do grupy
Dlaczego losowy split rekordów byłby niewystarczający, nawet gdy żaden tekst nie jest identyczny?